# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# The metadata object is a custom ML Croissant metadata instance, so display info using attributes:
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Overview of record sets (tables), their fields, and field IDs
print("Available record sets (by @id and name):\n")
for rset in dataset.record_sets:
    print(f"  - @id: {rset['@id']} | name: {rset.get('name', '<no name>')}")

print("\nRecord set -> fields mapping:")
for rset in dataset.record_sets:
    print(f"\nRecord set @id: {rset['@id']}  (name: {rset.get('name', '<no name>')})")
    if 'field' in rset and rset['field']:
        for field in rset['field']:
            field_obj = dataset.field_by_id(field['@id'])
            print(f"    - Field @id: {field_obj['@id']}, name: {field_obj.get('name', '<no name>')}, dataType: {field_obj.get('dataType', '<unknown>')}")
    else:
        print("    (No fields declared)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify available record sets (by @id) for loading
record_sets_ids = [rset['@id'] for rset in dataset.record_sets]
print("Selected record sets for extraction:")
print(record_sets_ids)

dataframes = {}
for record_set_id in record_sets_ids:
    # Load all records for this record set using mlcroissant API
    print(f"Loading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create dataframe if there is data
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"  No data found for record set {record_set_id}")

# Print sample columns of first record set with data
first_with_data = None
for rsid in record_sets_ids:
    if rsid in dataframes and not dataframes[rsid].empty:
        first_with_data = rsid
        break
if first_with_data:
    print(f"\nColumns in record set {first_with_data}:")
    print(dataframes[first_with_data].columns.tolist())
    display(dataframes[first_with_data].head())
else:
    print("No record set with available data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick the first record set with data and a likely numeric field for EDA.
record_set_id = first_with_data
if record_set_id is None:
    print("No record set with data available for EDA.")
else:
    df = dataframes[record_set_id]
    print(f"Performing EDA on record set: {record_set_id}")
    # Try to heuristically pick a numeric field (float or int based on metadata)
    numeric_fields = []
    rset = next(r for r in dataset.record_sets if r['@id'] == record_set_id)
    if 'field' in rset:
        for field in rset['field']:
            field_obj = dataset.field_by_id(field['@id'])
            dt = field_obj.get('dataType', '').lower()
            if dt in ['float', 'integer', 'number']:
                # Prefer field @id and 'name' for clarity
                fid = field_obj['@id']
                col_name = field_obj.get('name', fid)
                # Require that the column exists
                if col_name in df.columns:
                    numeric_fields.append(col_name)
    if not numeric_fields:
        # Fallback: Look for numeric columns in the actual dataframe
        numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()

    if numeric_fields:
        numeric_field = numeric_fields[0]  # Pick first one
        print(f"Selected numeric field for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
        # Ensure field is numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\n{numeric_field} normalized (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a non-numeric field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if record_set_id is not None and numeric_fields:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if available
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.